# Week 7: Self-Balancing AVL Trees (Rotations & Height Proof)

**Z5007 Programming and Data Structures (MTech DSAI Core) · IIT Madras Zanzibar**

LL, RR, LR, and RL rotations, AVL deletion, and Fibonacci-tree height verification.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/inyalala/iitmz-teaching-site/blob/main/courses/z5007/week-07/labs/W07_Self_Balancing_AVL_Trees.ipynb)

Run cells top to bottom (`Runtime > Run all`). Every algorithm is implemented from first principles in Python and verified with assertions.

## 1. AVL Node, Rotations (LL, RR, LR, RL) & Balance Invariant

Every AVL node caches its subtree height so `balance_factor(u) = height(u.left) - height(u.right)` runs in $O(1)$. Any $\pm 2$ imbalance repairs in $O(1)$ rotations.

In [1]:
import math
import random


class AVLNode:
    def __init__(self, key):
        self.key = key
        self.left = None
        self.right = None
        self.height = 0


def height(n):
    return -1 if n is None else n.height


def update_height(n):
    n.height = 1 + max(height(n.left), height(n.right))


def bf(n):
    return 0 if n is None else height(n.left) - height(n.right)


def rotate_right(z):
    y = z.left
    z.left, y.right = y.right, z
    update_height(z); update_height(y)
    return y


def rotate_left(z):
    y = z.right
    z.right, y.left = y.left, z
    update_height(z); update_height(y)
    return y


def rebalance(node):
    update_height(node)
    b = bf(node)
    if b > 1:
        if bf(node.left) < 0:
            node.left = rotate_left(node.left)
        return rotate_right(node)
    if b < -1:
        if bf(node.right) > 0:
            node.right = rotate_right(node.right)
        return rotate_left(node)
    return node


def avl_insert(node, key):
    if node is None:
        return AVLNode(key)
    if key < node.key:
        node.left = avl_insert(node.left, key)
    elif key > node.key:
        node.right = avl_insert(node.right, key)
    else:
        return node
    return rebalance(node)


def avl_delete(node, key):
    if node is None:
        return None
    if key < node.key:
        node.left = avl_delete(node.left, key)
    elif key > node.key:
        node.right = avl_delete(node.right, key)
    else:
        if node.left is None: return node.right
        if node.right is None: return node.left
        succ = node.right
        while succ.left:
            succ = succ.left
        node.key = succ.key
        node.right = avl_delete(node.right, succ.key)
    return rebalance(node)


def inorder(n):
    return [] if n is None else inorder(n.left) + [n.key] + inorder(n.right)


root = None
for k in [60, 30, 40, 80, 70, 90, 75]:
    root = avl_insert(root, k)
root = avl_delete(root, 30)
print("Worksheet inorder after delete 30:", inorder(root), "| root:", root.key)
assert inorder(root) == [40, 60, 70, 75, 80, 90]


Worksheet inorder after delete 30: [40, 60, 70, 75, 80, 90] | root: 70


## 2. Fibonacci Height Proof & Sorted-Input Benchmark

Because the minimum number of nodes in an AVL tree of height $h$ satisfies $N(h) = 1 + N(h-1) + N(h-2) = F_{h+3} - 1$, height never exceeds $\approx 1.44 \log_2(n + 2)$ even on strictly sorted input.

In [2]:
def verify_avl(node):
    if node is None:
        return True, -1
    ok_l, hl = verify_avl(node.left)
    ok_r, hr = verify_avl(node.right)
    return ok_l and ok_r and abs(hl - hr) <= 1 and node.height == 1 + max(hl, hr), node.height


for n in (100, 500, 1000, 2000):
    t = None
    for k in range(n):
        t = avl_insert(t, k)
    ok, h = verify_avl(t)
    bound = 1.44 * math.log2(n + 2) - 0.328
    assert ok and h <= bound
    print(f"n={n:>4} sorted inserts -> AVL height={h:>2} (worst-case bound {bound:.2f}, plain BST would be {n-1})")


n= 100 sorted inserts -> AVL height= 6 (worst-case bound 9.28, plain BST would be 99)
n= 500 sorted inserts -> AVL height= 8 (worst-case bound 12.59, plain BST would be 499)
n=1000 sorted inserts -> AVL height= 9 (worst-case bound 14.03, plain BST would be 999)
n=2000 sorted inserts -> AVL height=10 (worst-case bound 15.46, plain BST would be 1999)
